In [1]:
import os

# Logic to switch between colab and local IDE
is_colab = False

main_dir = os.curdir

print(f"Main Directory: {main_dir}")

# Store the clean, downloaded BERT model here
base_model_path = os.path.join(main_dir, "model", "bertweet_base_local")
# Store your training checkpoints here
checkpoint_dir = os.path.join(main_dir, "model", "checkpoints")
# Store the datasets here
dataset_dir = os.path.join(main_dir, "data")

os.makedirs(base_model_path, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(dataset_dir, exist_ok=True)

print(f"Base model will be stored at: {base_model_path}")
print(f"Checkpoints will be saved to: {checkpoint_dir}")
print(f"Datasets will be stored to: {dataset_dir}")


Main Directory: .
Base model will be stored at: ./model/bertweet_base_local
Checkpoints will be saved to: ./model/checkpoints
Datasets will be stored to: ./data


## Load Datasets

In [2]:
from datasets import load_dataset
from huggingface_hub import list_repo_files, hf_hub_download
import os

# Reuse dataset_dir from the setup cell
os.makedirs(dataset_dir, exist_ok=True)

revision = "refs/convert/parquet"
subsets = ["pubhealth_source", "pubhealth_bigbio_pairs"]

repo_files = list_repo_files(
    "bigbio/pubhealth",
    repo_type="dataset",
    revision=revision,
)

def download_split_files(subset_name):
    parquet_files = [
        f for f in repo_files
        if f.startswith(subset_name) and f.endswith(".parquet")
    ]
    if not parquet_files:
        raise RuntimeError(
            f"No parquet files found for subset {subset_name} at {revision}"
        )

    split_files = {"train": [], "validation": [], "test": []}
    for f in parquet_files:
        if "/train/" in f:
            split = "train"
        elif "/validation/" in f:
            split = "validation"
        elif "/test/" in f:
            split = "test"
        else:
            split = "train"

        local_path = os.path.join(dataset_dir, f)
        if os.path.exists(local_path):
            split_files[split].append(local_path)
        else:
            split_files[split].append(
                hf_hub_download(
                    repo_id="bigbio/pubhealth",
                    repo_type="dataset",
                    filename=f,
                    revision=revision,
                    local_dir=dataset_dir,
                    local_dir_use_symlinks=False,
                )
            )

    # Remove empty splits to avoid errors
    return {k: v for k, v in split_files.items() if v}

source_files = download_split_files("pubhealth_source")
print(f"Source files: {source_files}")
pair_files = download_split_files("pubhealth_bigbio_pairs")
print(f"Pair files: {pair_files}")

# Load each subset separately (different schemas)
source_ds = load_dataset("parquet", data_files=source_files)
pair_ds = load_dataset("parquet", data_files=pair_files)

index = 3
display(source_ds["train"][index])
display(pair_ds["train"][index])

Source files: {'train': ['./data/pubhealth_source/train/0000.parquet'], 'validation': ['./data/pubhealth_source/validation/0000.parquet'], 'test': ['./data/pubhealth_source/test/0000.parquet']}
Pair files: {'train': ['./data/pubhealth_bigbio_pairs/train/0000.parquet'], 'validation': ['./data/pubhealth_bigbio_pairs/validation/0000.parquet'], 'test': ['./data/pubhealth_bigbio_pairs/test/0000.parquet']}


{'claim_id': '10166',
 'claim': 'Study: Vaccine for Breast, Ovarian Cancer Has Potential',
 'date_published': 'November 8, 2011',
 'explanation': 'While the story does many things well, the overall framing of the story is that the vaccine “shows promise,” when the evidence actually points in the other direction. Because only one patient in the study remains cancer free and because that patient may very well have benefited from an earlier cancer vaccine and other complicating factors, we question the decision to write this story in the first place. Right now, there more than 10,000 cancer-related clinical trials recruiting patients. Cancer has foiled scientists repeatedly with treatments that initially seemed promising in the laboratory or in a very small group of people and later proved unworkable on a larger scale. It’s a difficult task — but a crucial one — for reporters to ask tough questions of the evidence and a wide range of sources before deciding whether one of these thousands 

{'id': '3',
 'document_id': '10166',
 'text_1': 'Study: Vaccine for Breast, Ovarian Cancer Has Potential',
 'text_2': 'While the story does many things well, the overall framing of the story is that the vaccine “shows promise,” when the evidence actually points in the other direction. Because only one patient in the study remains cancer free and because that patient may very well have benefited from an earlier cancer vaccine and other complicating factors, we question the decision to write this story in the first place. Right now, there more than 10,000 cancer-related clinical trials recruiting patients. Cancer has foiled scientists repeatedly with treatments that initially seemed promising in the laboratory or in a very small group of people and later proved unworkable on a larger scale. It’s a difficult task — but a crucial one — for reporters to ask tough questions of the evidence and a wide range of sources before deciding whether one of these thousands of experimental treatment op

## Test Few False and True Claims

## Load Intel/misinformation-guard Dataset

In [3]:
import os

intel_ds = load_dataset("Intel/misinformation-guard")

int_to_text_map = {
    0: "false",
    1: "mostly_true",
    2: "partially_true",
    3: "true"
}

intel_ds = intel_ds.map(
    lambda x: {"label_text": int_to_text_map[x["label"]]}
)

intel_save_path = os.path.join(dataset_dir, "intel_misinformation_guard")
os.makedirs(intel_save_path, exist_ok=True)
intel_ds.save_to_disk(intel_save_path)
print(f"Intel dataset saved to: {intel_save_path}")

display(intel_ds["train"][1])

Saving the dataset (0/1 shards):   0%|          | 0/26812 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6703 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8379 [00:00<?, ? examples/s]

Intel dataset saved to: ./data/intel_misinformation_guard


{'text': 'Vaccinating children against COVID-19 can reduce the risk of severe illness and hospitalization, but the vaccine does not guarantee complete immunity against the virus.',
 'reasoning': 'This statement is mostly true because numerous studies have shown that COVID-19 vaccines are highly effective in preventing severe illness and hospitalization in children. However, no vaccine is 100% effective, and individuals may still contract the virus even after vaccination. The statement is largely accurate but may lack nuance regarding the limitations of vaccine efficacy.',
 'label': 1,
 'model': 'meta-llama/Meta-Llama-3.1-8B-Instruct',
 'label_text': 'mostly_true'}

## Setup API Environments

In [22]:
import os
from openai import OpenAI
from typing import Dict, List

openai_api_key = os.getenv("OPENAI_API_KEY")
deepinfra_api_key = os.getenv("DEEPINFRA_TOKEN")
gemini_api_key = os.getenv("GEMINI_API_KEY")
featherless_api_key = os.getenv("FEATHERLESS_API_KEY")

gen_params = {
    "model": "gpt-4o-mini",
    "temperature": 0.0,
    "max_tokens": 100,
    "top_p": 1.0,
    "frequency_penalty": 0.0,
    "presence_penalty": 0.0,
    "reasoning": "low"
}

def generate_openai_response(prompt: str|List[Dict], *, openai_api_key=openai_api_key, **gen_params):
    client = OpenAI()

    model = gen_params.get("model", "gpt-4o-mini")
    reasoning_param = gen_params.get("reasoning", False)
    if reasoning_param:
        reasoning_param = {"effort": reasoning_param}
    else:
        reasoning_param = None

    if isinstance(prompt, str):
        prompt = [{"role": "user", "content": prompt}]
    else:
        prompt = prompt

    response = client.responses.create(
        model=model,
        reasoning=reasoning_param,
        input=prompt,
        **gen_params
    )
    return response.output_text

def generate_deepinfra_response(prompt: str|List[Dict], *, deepinfra_api_key=deepinfra_api_key, stream=False, **gen_params):
    client = OpenAI(api_key=deepinfra_api_key, base_url="https://api.deepinfra.com/v1/openai")

    chat_completion = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=[{"role": "user", "content": "Hello"}],
        stream=stream,
    )

    if stream:
        chunks = []
        for event in chat_completion:
            delta = event.choices[0].delta.content
            if delta:  # avoid printing None
                print(delta, end="", flush=True)
                chunks.append(delta)
        print()
        return "".join(chunks)

    return chat_completion.choices[0].message.content


In [3]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
  api_key=os.getenv("GEMINI_API_KEY"),
  base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

models = client.models.list()
for model in models:
  print(model.id)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/aqa
models/